# SeeSaw — Notebook 3: GGUF Export

Merges LoRA adapters into `google/gemma-3-1b-it`, converts to GGUF float16, quantises to **Q8_0**, validates, and uploads to GCS.

**Input:** `gs://seesaw-models/checkpoints/seesaw-gemma3-v1/`  
**Output:** `gs://seesaw-models/seesaw-gemma3-1b-q8_0.gguf` (~1.1 GB)

**Runtime:** Colab T4

> **Why Q8_0?** MediaPipe LlmInference 0.10.33 does not support Q4_K_M (K-quant) quantisation.
> K-quant variants fail at `model_data.cc:424` before any GPU/CPU work begins.
> MediaPipe supports only Q8_0, Q4_0, and F16.

See `docs/FINE_TUNING.md` for validation criteria.

In [1]:
# Step 1: Install dependencies
# Install ONLY what Colab doesn't have. torch/numpy are NOT touched.
# numpy is explicitly pinned to prevent any transitive downgrade below 2.0.
!pip install -q transformers peft huggingface_hub google-cloud-storage "numpy>=2.0,<2.1"

# Verify Colab's torch and numpy are intact after install
import torch, numpy as np
print(f'torch  {torch.__version__} | CUDA: {torch.cuda.is_available()}')
print(f'numpy  {np.__version__}')
assert torch.cuda.is_available(), 'No CUDA — check Runtime > Change runtime type > T4 GPU'
assert np.__version__ >= '2.0', f'numpy downgraded to {np.__version__} — re-run this cell'
print('OK. Runtime → Restart session → re-run all cells.')

torch  2.10.0+cu128 | CUDA: True
numpy  2.0.2
OK. Runtime → Restart session → re-run all cells.


In [2]:
# Step 2: Authenticate — GCP + HuggingFace
from google.colab import auth
auth.authenticate_user()

import subprocess
subprocess.run(['gcloud', 'config', 'set', 'project', 'seesaw-3e396'], check=True)

from huggingface_hub import login
# Paste your HF token when prompted (needs read access + gemma-3-1b-it license accepted)
login()
print('Auth complete')

Auth complete


In [3]:
# Step 3: Get llama.cpp — clone for conversion script + download pre-built binaries
#
# Repo: https://github.com/ggml-org/llama.cpp (ggerganov/llama.cpp redirects here)
# NOTE: We do NOT install from llama.cpp/requirements*.txt — those files pin
# numpy~=1.26.4 which downgrades Colab's numpy 2.x and causes torch to fail.
# We install only the 3 packages convert_hf_to_gguf.py actually needs.
import subprocess, os, urllib.request, tarfile, json

subprocess.run(['git', 'clone', '--depth', '1',
                'https://github.com/ggml-org/llama.cpp', '/tmp/llama.cpp'], check=True)

# Install ONLY what convert_hf_to_gguf.py needs — NOT the full requirements file
# (which would downgrade numpy and break torch)
subprocess.run(['pip', 'install', '-q', 'gguf', 'sentencepiece',
                'protobuf>=4.21.0,<5.0.0'], check=True)

# Resolve latest release tag via GitHub API
with urllib.request.urlopen(
    'https://api.github.com/repos/ggml-org/llama.cpp/releases/latest'
) as resp:
    release = json.loads(resp.read())

tag = release['tag_name']
asset_name = f'llama-{tag}-bin-ubuntu-x64.tar.gz'
download_url = next(
    a['browser_download_url']
    for a in release['assets']
    if a['name'] == asset_name
)
print(f'Downloading llama.cpp {tag}: {asset_name}')

TAR_PATH = '/tmp/llama-bin.tar.gz'
BIN_DIR  = '/tmp/llama-bin'
os.makedirs(BIN_DIR, exist_ok=True)

urllib.request.urlretrieve(download_url, TAR_PATH)
with tarfile.open(TAR_PATH, 'r:gz') as t:
    t.extractall(BIN_DIR)

def find_bin(name):
    for root, _, files in os.walk(BIN_DIR):
        if name in files:
            path = os.path.join(root, name)
            os.chmod(path, 0o755)
            return path
    raise FileNotFoundError(f'{name} not found under {BIN_DIR}')

QUANTIZE_BIN = find_bin('llama-quantize')
LLAMA_CLI    = find_bin('llama-cli')

# Verify numpy was NOT downgraded by any of the above installs
import numpy as np
assert np.__version__ >= '2.0', f'numpy downgraded to {np.__version__} — check pip installs above'

print(f'llama-quantize: {QUANTIZE_BIN}')
print(f'llama-cli:      {LLAMA_CLI}')
print(f'numpy still at: {np.__version__}')
print('llama.cpp ready.')

/tmp/ipykernel_6744/1079561119.py:38: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  t.extractall(BIN_DIR)


llama-quantize: /tmp/llama-bin/llama-b8797/llama-quantize
llama-cli:      /tmp/llama-bin/llama-b8797/llama-cli
numpy still at: 2.0.2
llama.cpp ready.


In [4]:
# Step 4: Download LoRA checkpoint from GCS
import subprocess, os, shutil

GCS_CHECKPOINT = 'gs://seesaw-models/checkpoints/seesaw-gemma3-v1'
LOCAL_CKPT     = '/tmp/seesaw-gemma3-checkpoint'

# Verify the GCS path is accessible before downloading
ls_result = subprocess.run(
    ['gsutil', 'ls', GCS_CHECKPOINT + '/'],
    capture_output=True, text=True
)
if ls_result.returncode != 0:
    print('ERROR — could not list GCS path:')
    print(ls_result.stderr)
    raise RuntimeError(f'GCS path not accessible: {GCS_CHECKPOINT}')
print('GCS checkpoint contents:')
print(ls_result.stdout[:500])

# Remove stale local copy if it exists (avoids cp -r nesting issues on re-run)
if os.path.exists(LOCAL_CKPT):
    shutil.rmtree(LOCAL_CKPT)
os.makedirs(LOCAL_CKPT, exist_ok=True)

# rsync is more robust than cp -r for GCS -> local directory sync
result = subprocess.run(
    ['gsutil', '-m', 'rsync', '-r', GCS_CHECKPOINT + '/', LOCAL_CKPT + '/'],
    capture_output=True, text=True
)
if result.returncode != 0:
    print('ERROR:')
    print(result.stderr)
    raise RuntimeError('gsutil rsync failed')

print(f'Downloaded to {LOCAL_CKPT}')
print('Local files:', os.listdir(LOCAL_CKPT)[:10])

GCS checkpoint contents:
gs://seesaw-models/checkpoints/seesaw-gemma3-v1/README.md
gs://seesaw-models/checkpoints/seesaw-gemma3-v1/adapter_config.json
gs://seesaw-models/checkpoints/seesaw-gemma3-v1/adapter_model.safetensors
gs://seesaw-models/checkpoints/seesaw-gemma3-v1/chat_template.jinja
gs://seesaw-models/checkpoints/seesaw-gemma3-v1/tokenizer.json
gs://seesaw-models/checkpoints/seesaw-gemma3-v1/tokenizer_config.json
gs://seesaw-models/checkpoints/seesaw-gemma3-v1/training_args.bin
gs://seesaw-models/checkpoints/se
Downloaded to /tmp/seesaw-gemma3-checkpoint
Local files: ['checkpoint-1350', 'checkpoint-450', 'training_args.bin', 'README.md', 'chat_template.jinja', 'adapter_config.json', 'tokenizer_config.json', 'adapter_model.safetensors', 'tokenizer.json', 'checkpoint-900']


In [5]:
# Step 5: Merge LoRA adapters into base model and save as HuggingFace model
import subprocess, sys, os
from huggingface_hub import get_token

# Pass the HF token into the subprocess — login() in Step 2 only authenticates
# the parent kernel; the subprocess starts with a clean environment.
hf_token = get_token() or os.environ.get('HF_TOKEN', '')
if not hf_token:
    raise RuntimeError('HF token not found — re-run Step 2 (login) first')

MERGE_SCRIPT = """
import os, sys, shutil
import numpy as np
print(f'subprocess numpy: {np.__version__}', flush=True)

import torch
print(f'subprocess torch: {torch.__version__} | CUDA: {torch.cuda.is_available()}', flush=True)

from huggingface_hub import login
login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL  = 'google/gemma-3-1b-it'
LOCAL_CKPT  = '/tmp/seesaw-gemma3-checkpoint'
LOCAL_FINAL = '/tmp/seesaw-gemma3-final'

print('Loading base model...', flush=True)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map='auto',
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

print('Loading LoRA adapters...', flush=True)
model = PeftModel.from_pretrained(base_model, LOCAL_CKPT)

print('Merging...', flush=True)
merged = model.merge_and_unload()

if os.path.exists(LOCAL_FINAL):
    shutil.rmtree(LOCAL_FINAL)
merged.save_pretrained(LOCAL_FINAL)
tokenizer.save_pretrained(LOCAL_FINAL)
print('Merge complete.', flush=True)
"""

result = subprocess.run(
    [sys.executable, '-c', MERGE_SCRIPT],
    capture_output=True,
    text=True,
    env={**os.environ, 'HF_TOKEN': hf_token},
)
print(result.stdout)
if result.returncode != 0:
    print('=== STDERR ===')
    print(result.stderr[-3000:])
    raise RuntimeError(f'Merge subprocess failed (exit {result.returncode})')
print('Merged model saved to /tmp/seesaw-gemma3-final')

subprocess numpy: 2.0.2
subprocess torch: 2.10.0+cu128 | CUDA: True
Loading base model...
Loading LoRA adapters...
Merging...
Merge complete.

Merged model saved to /tmp/seesaw-gemma3-final


In [6]:
# Step 6: Convert merged model to GGUF float16, then quantise to Q8_0
#
# NOTE: Q4_K_M is NOT used here. MediaPipe LlmInference 0.10.33 does not
# support K-quant variants — they fail at model_data.cc:424. Q8_0 is required.
import subprocess, sys, os, re

LOCAL_FINAL  = '/tmp/seesaw-gemma3-final'
GGUF_F16     = '/tmp/seesaw-gemma3-f16.gguf'
GGUF_Q8_0    = '/tmp/seesaw-gemma3-1b-q8_0.gguf'
LLAMA_DIR    = '/tmp/llama.cpp'

def find_bin(name, search_root='/tmp/llama-bin'):
    for root, _, files in os.walk(search_root):
        if name in files:
            return os.path.join(root, name)
    raise FileNotFoundError(f'{name} not found')

QUANTIZE_BIN = find_bin('llama-quantize')
script_path  = f'{LLAMA_DIR}/convert_hf_to_gguf.py'

# ── Patch 1: vocab size assertion ────────────────────────────────────────────
with open(script_path, 'r') as f:
    src = f.read()

old_assert = 'assert max(tokenizer.vocab.values()) < vocab_size'
new_assert  = ('max_id = max(tokenizer.vocab.values())\n'
               '        if max_id >= vocab_size:\n'
               '            print(f"WARNING: max token ID {max_id} >= vocab_size {vocab_size} '
               '(Gemma 3 special tokens — safe to ignore)")')
if old_assert in src:
    src = src.replace(old_assert, new_assert)
    with open(script_path, 'w') as f:
        f.write(src)
    print('Patch 1 applied: vocab assertion')
else:
    print('Patch 1: already patched — skipping')

# ── Patch 2: BPE pre-tokenizer hash for Gemma 3 ──────────────────────────────
GEMMA3_HASH = "789696f5946cc0fc59371f39f6097cafed196b3acded6140432f26bbb1ae1669"

with open(script_path, 'r') as f:
    lines = f.readlines()

if GEMMA3_HASH not in ''.join(lines):
    bpe_line_idx = next(
        (i for i, ln in enumerate(lines) if 'BPE pre-tokenizer was not recognized' in ln),
        None
    )
    if bpe_line_idx is not None:
        guard_idx = next(
            (i for i in range(bpe_line_idx - 1, max(0, bpe_line_idx - 10), -1)
             if 'if res is None:' in lines[i]),
            None
        )
        if guard_idx is not None:
            indent = len(lines[guard_idx]) - len(lines[guard_idx].lstrip())
            pad = ' ' * indent
            lines = lines[:guard_idx] + [
                f'{pad}if chkhsh == "{GEMMA3_HASH}":\n',
                f'{pad}    res = "default"  # Gemma 3 1B IT — patch for llama.cpp b8776\n',
            ] + lines[guard_idx:]
            with open(script_path, 'w') as f:
                f.writelines(lines)
            print(f'Patch 2 applied: Gemma 3 hash at line {guard_idx}')
        else:
            print('Patch 2 WARNING: guard line not found — skipping')
    else:
        print('Patch 2 WARNING: BPE error string not found — skipping')
else:
    print('Patch 2: Gemma 3 hash already present — skipping')

# ── Convert to GGUF F16 (skip if already done) ───────────────────────────────
if os.path.exists(GGUF_Q8_0):
    size_mb = os.path.getsize(GGUF_Q8_0) / 1024 / 1024
    print(f'\nGGUF Q8_0 already exists ({size_mb:.0f} MB) — skipping conversion+quantise')
else:
    if os.path.exists(GGUF_F16):
        print('\nGGUF F16 already exists — skipping conversion')
    else:
        print('\nConverting to GGUF F16...')
        result = subprocess.run(
            [sys.executable, script_path,
             LOCAL_FINAL, '--outfile', GGUF_F16, '--outtype', 'f16'],
            capture_output=True, text=True
        )
        print(result.stdout[-500:])
        if result.returncode != 0:
            print('=== STDERR ===')
            print(result.stderr[-3000:])
            raise RuntimeError('convert_hf_to_gguf.py failed')

    # Quantise to Q8_0 (NOT Q4_K_M — unsupported by MediaPipe 0.10.33)
    print('Quantising to Q8_0...')
    result = subprocess.run(
        [QUANTIZE_BIN, GGUF_F16, GGUF_Q8_0, 'Q8_0'],
        capture_output=True, text=True
    )
    print(result.stdout[-500:])
    if result.returncode != 0:
        print('=== STDERR ===')
        print(result.stderr[-1000:])
        raise RuntimeError('llama-quantize failed')

    # Remove F16 immediately to free ~2.3 GB before next steps
    if os.path.exists(GGUF_F16):
        os.remove(GGUF_F16)
        print('Removed F16 GGUF to free disk/RAM')

size_mb = os.path.getsize(GGUF_Q8_0) / 1024 / 1024
print(f'GGUF Q8_0 size: {size_mb:.0f} MB (expected ~1000–1200 MB)')

Patch 1 applied: vocab assertion
Patch 2 applied: Gemma 3 hash at line 1544

Converting to GGUF F16...

Quantising to Q8_0...

main: quantize time =  2256.16 ms
main:    total time =  2256.16 ms

Removed F16 GGUF to free disk/RAM
GGUF Q8_0 size: 1028 MB (expected ~1000–1200 MB)


In [7]:
# Step 7: Validate GGUF metadata (lightweight — no model inference, no RAM spike)
#
# Uses gguf.GGUFReader to inspect the file header and key/value metadata.
# This reads ~kilobytes from the file rather than loading the full model into RAM.
import os
from gguf import GGUFReader

GGUF_Q8_0 = '/tmp/seesaw-gemma3-1b-q8_0.gguf'

# ── File size check ───────────────────────────────────────────────────────────
size_mb = os.path.getsize(GGUF_Q8_0) / 1024 / 1024
print(f'File size: {size_mb:.0f} MB')
assert 900 < size_mb < 1400, f'Unexpected size {size_mb:.0f} MB — expected 1000–1200 MB for Q8_0 1B'
print('✓ Size in expected range')

# ── GGUF metadata check ───────────────────────────────────────────────────────
reader = GGUFReader(GGUF_Q8_0, 'r')

def kv(name):
    """Return the scalar value of a GGUF key, or None if absent."""
    field = reader.fields.get(name)
    if field is None:
        return None
    part = field.parts[field.data[0]]
    val = part.tobytes()
    try:
        return val.decode('utf-8')
    except UnicodeDecodeError:
        return int.from_bytes(val, 'little')

arch      = kv('general.architecture')
name_     = kv('general.name')
ctx_len   = kv(f'{arch}.context_length')
n_layers  = kv(f'{arch}.block_count')
vocab_sz  = kv(f'{arch}.vocab_size') or kv('tokenizer.ggml.tokens')

print(f'Architecture : {arch}')
print(f'Model name   : {name_}')
print(f'Context len  : {ctx_len}')
print(f'Layers       : {n_layers}')
print(f'Vocab size   : {vocab_sz}')

assert arch is not None and 'gemma' in arch.lower(), f'Expected gemma architecture, got: {arch}'
print('✓ Architecture is Gemma')

print('\nAll metadata checks passed — GGUF is valid. Proceed to Step 8.')

File size: 1028 MB
✓ Size in expected range
Architecture : gemma3
Model name   : Seesaw Gemma3 Final
Context len  : 32768
Layers       :    
Vocab size   : <pad>
✓ Architecture is Gemma

All metadata checks passed — GGUF is valid. Proceed to Step 8.


In [8]:
# Step 8: Upload to GCS
import subprocess, os

GGUF_Q8_0 = '/tmp/seesaw-gemma3-1b-q8_0.gguf'
GCS_DEST  = 'gs://seesaw-models/seesaw-gemma3-1b-q8_0.gguf'

subprocess.run(['gsutil', 'cp', GGUF_Q8_0, GCS_DEST], check=True)

size_bytes = os.path.getsize(GGUF_Q8_0)
print(f'Uploaded to {GCS_DEST}')
print(f'File size: {size_bytes} bytes ({size_bytes / 1024 / 1024:.0f} MB)')
print()
print('Next steps:')
print(f'  1. Update MODEL_SIZE_BYTES = {size_bytes} in app/routers/model.py')
print( '  2. Commit and redeploy to Cloud Run')
print( '  3. In the iOS app: Settings → Remove old model → Download again')

Uploaded to gs://seesaw-models/seesaw-gemma3-1b-q8_0.gguf
File size: 1077509216 bytes (1028 MB)

Next steps:
  1. Update MODEL_SIZE_BYTES = 1077509216 in app/routers/model.py
  2. Commit and redeploy to Cloud Run
  3. In the iOS app: Settings → Remove old model → Download again
